In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.utils import shuffle 
  
# ========================================================= 
# 1. LOAD DATA & ROBUST CLEANING 
# ========================================================= 
print("Loading data...") 
try: 
    # Attempt standard load 
    df = pd.read_csv('train.csv') 
except pd.errors.ParserError: 
    print("Standard parser failed due to messy Wikipedia text. Switching to Python engine...") 
    # Fallback for messy HTML/quotes in the Kaggle dataset 
    df = pd.read_csv('train.csv', engine='python', on_bad_lines='skip') 
  
# Drop empty rows and calculate word count for EDA 
df = df.dropna(subset=['comment_text']) 
df['word_count'] = df['comment_text'].apply(lambda x: len(str(x).split())) 
  
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'] 
  
# ========================================================= 
# 2. EDA VISUALIZATIONS (For your Final Report) 
# ========================================================= 
print("Generating EDA Visuals...") 
  
# A. Class Imbalance Plot 
plt.figure(figsize=(10, 6)) 
counts = df[label_cols].sum().sort_values(ascending=False) 
ax = sns.barplot(x=counts.index, y=counts.values, palette='viridis') 
plt.title('Real-World Distribution: High Class Imbalance') 
plt.ylabel('Total Count in Dataset') 
# Add numbers on top of bars for clarity 
for p in ax.patches: 
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5), 
                textcoords='offset points') 
plt.show() 
  
# B. Correlation Matrix 
plt.figure(figsize=(8, 6)) 
corr_matrix = df[label_cols].corr() 
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=0, vmax=1) 
plt.title('Toxicity Label Correlations') 
plt.show() 
  
# ========================================================= 
# 3. "STRESS TEST" SAMPLING (WITH MINORITY RESERVE) 
# ========================================================= 
print("\nExecuting Custom Stress-Test Sampling...") 
  
# Identify if a comment is completely clean 
df['is_clean'] = (df[label_cols].sum(axis=1) == 0).astype(int) 
  
# Separate pools 
clean_pool = df[df['is_clean'] == 1].copy() 
toxic_pool = df[df['is_clean'] == 0].copy() 
  
# A. Isolate EVERY single rare comment 
all_threats = toxic_pool[toxic_pool['threat'] == 1] 
all_hate = toxic_pool[(toxic_pool['identity_hate'] == 1) & (toxic_pool['threat'] == 0)] 
all_severe = toxic_pool[(toxic_pool['severe_toxic'] == 1) & (toxic_pool['threat'] == 0) & (toxic_pool['identity_hate'] == 0)] 
  
# B. Reserve exactly 20 of each for the Few-Shot Prompting Pool 
prompt_threats = all_threats.sample(n=20, random_state=42) 
prompt_hate = all_hate.sample(n=20, random_state=42) 
prompt_severe = all_severe.sample(n=20, random_state=42) 
  
# C. Remove the reserved examples so they strictly go into the test pool 
test_threats = all_threats.drop(prompt_threats.index) 
test_hate = all_hate.drop(prompt_hate.index) 
test_severe = all_severe.drop(prompt_severe.index) 
  
# D. Pad the rest of the toxic test set (aiming for exactly 4,000 toxic comments) 
current_test_toxic_count = len(test_threats) + len(test_hate) + len(test_severe) 
needed_toxic = 4000 - current_test_toxic_count 
  
# Exclude ALL rare items (both test and prompt reserves) before sampling the standard toxic ones 
remaining_toxic = toxic_pool[~toxic_pool.index.isin(all_threats.index.union(all_hate.index).union(all_severe.index))] 
standard_toxic_sample = remaining_toxic.sample(n=needed_toxic, random_state=42) 
  
# Combine for the final toxic half of the test set 
final_toxic_test = pd.concat([test_threats, test_hate, test_severe, standard_toxic_sample]) 
  
# E. Match with an equal number of CLEAN comments (50/50 Split) 
final_clean_test = clean_pool.sample(n=len(final_toxic_test), random_state=42) 
  
# Combine and shuffle to create the final Evaluation Set 
eval_set = pd.concat([final_toxic_test, final_clean_test]) 
eval_set = shuffle(eval_set, random_state=42).reset_index(drop=True) 
  
# ========================================================= 
# 4. CREATE THE FEW-SHOT POOL & SAVE (FIXED: ZERO LEAKAGE) 
# ========================================================= 
print("\nGenerating Leak-Free Few-Shot Pool...") 
  
# All 28 manual IDs you want excluded from the RAG pool 
manual_ids_to_remove = [ 
    "99412dd64aa5ab0d", "6ac5659863a55d02", "f28fe9e4a08efc8a", "b2fd96124498d96d",
    "86afedcac341a144", "4874125610095abf", "3ccc105614ccef54", "6529a467f256647f",
    "6df40958126e56f7", "f66e653a534aa331", "7b72d20b1b0fc864", "4de36e202b2a1be1",
    "9c5673edc3e2841f", "671df5c336134e47", "409b72640b67f366", "5b5c9379e61547b2",
    "dc007ad85fe628fc", "044f836b8be214a1", "3145558c11718e97", "48f55b8e13d9df68",
    "26ec76fc345b83ad", "831499df53e45654", "28aa9dfdb6094c88", "53c1df4998ffa274",
    "6a9eb57a1494f576",
    "474a3b6a20f3d6b7",  
    "2a41d16dc69ca681",  
    "4e9e891958d2422c" 
] 
  
# 1. First, mathematically subtract the 8K eval set from the entire dataset using Kaggle 'id' 
few_shot_pool = df[~df['id'].isin(eval_set['id'])].copy() 
  
# 2. Second, mathematically subtract your manual IDs so they don't get pulled by RAG 
few_shot_pool = few_shot_pool[~few_shot_pool['id'].isin(manual_ids_to_remove)].copy() 
  
# Save to CSV files 
eval_set.to_csv('stress_test_eval_set.csv', index=False) 
few_shot_pool.to_csv('few_shot_pool_fixed.csv', index=False) 
  
# ========================================================= 
# 5. FINAL SANITY CHECK / OUTPUT 
# ========================================================= 
print("\n=== SAMPLING COMPLETE ===") 
print(f"Total Evaluation Set Size: {len(eval_set)} rows") 
print(f" -> Clean Comments: {len(final_clean_test)}") 
print(f" -> Toxic Comments: {len(final_toxic_test)}") 
print("\nRare Class Support in EVAL SET (What the model will be graded on):") 
print(f" -> Threats: {eval_set['threat'].sum()}") 
print(f" -> Identity Hate: {eval_set['identity_hate'].sum()}") 
print(f" -> Severe Toxic: {eval_set['severe_toxic'].sum()}") 
  
print("\nMinority classes successfully SECURED in FEW-SHOT POOL (For Prompt Engineering):") 
print(f" -> Threats: {len(prompt_threats)}") 
print(f" -> Identity Hate: {len(prompt_hate)}") 
print(f" -> Severe Toxic: {len(prompt_severe)}") 
print(f"Total Few-Shot Pool Size: {len(few_shot_pool)} rows (Leakage safely removed!)") 
  
# ========================================================= 
# 6. STRESS-TEST EVAL SET DISTRIBUTION VISUALIZATION 
# ========================================================= 
print("\nGenerating Stress-Test Distribution Chart...") 
  
plt.figure(figsize=(10, 6)) 
  
# We want to show the toxic labels PLUS the clean comments to prove the 50/50 split 
eval_counts = eval_set[label_cols + ['is_clean']].sum().sort_values(ascending=False) 
  
# Rename 'is_clean' to 'Clean' for a nicer chart label 
eval_counts = eval_counts.rename({'is_clean': 'Clean', 'severe_toxic': 'severe', 'identity_hate': 'hate'}) 
  
ax2 = sns.barplot(x=eval_counts.index, y=eval_counts.values, palette='magma') 
plt.title('Stress-Test Evaluation Set: Engineered Distribution for Safety Audit') 
plt.ylabel('Count in Eval Set') 
plt.xlabel('Toxicity Labels & Clean Comments') 
  
# Add numbers on top of bars for clarity 
for p in ax2.patches: 
    ax2.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5), 
                textcoords='offset points') 
  
# Save the chart so you can easily put it in your PDF report 
plt.savefig('stress_test_distribution.png', bbox_inches='tight') 
plt.show() 
  
print("Distribution chart saved as 'stress_test_distribution.png'!")